# Train UMOD ResNet34 U-Net on Colab

1. Upload the whole `umod-unet` project folder to Google Drive (for example `MyDrive/umod-unet`).
2. Open **this** notebook from Drive in Colab.
3. **Runtime → Change runtime type → T4 GPU**.
4. Run the cells in order. Keep the tab open while it trains (~40–60 min).

If Colab disconnects, reconnect, set the GPU again, and rerun from the top. Training resumes from `last.pt` on Drive.

This is not a medical device.

In [ ]:
from pathlib import Path
from google.colab import drive
import torch, subprocess, sys, os

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

REPO = Path("/content/drive/MyDrive/umod-unet")
ZIP_PATH = Path("/content/drive/MyDrive/umod_data/ds1_download_test.zip")
DATA_DIR = Path("/content/umod")
RUN_DIR = Path("/content/drive/MyDrive/umod_runs/resnet34_unet")
DATA_URL = "https://rodare.hzdr.de/api/files/2d96466e-612d-4754-afcc-c65e05fc25da/ds1.zip"

assert (REPO / "umod_unet").is_dir(), f"Upload the umod-unet folder to Drive so this exists: {REPO}"
print("repo:", REPO)

In [ ]:
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
MIN_ZIP_BYTES = 1_000_000_000
if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size < MIN_ZIP_BYTES:
    part = ZIP_PATH.with_name(ZIP_PATH.name + ".part")
    print("Downloading ds1.zip (~2.3 GB) ...")
    result = subprocess.run(
        ["curl", "-fL", "--retry", "3", "--retry-delay", "10", "-C", "-", DATA_URL, "--output", str(part)]
    )
    size = part.stat().st_size if part.exists() else 0
    if result.returncode != 0 or size < MIN_ZIP_BYTES:
        raise RuntimeError(f"Download failed (code {result.returncode}, {size} bytes). Get ds1.zip from https://rodare.hzdr.de/record/2563")
    part.rename(ZIP_PATH)
print("zip:", ZIP_PATH, "GB:", round(ZIP_PATH.stat().st_size / 1e9, 2))

if not (DATA_DIR / "train").exists():
    DATA_DIR.mkdir(exist_ok=True)
    print("Unzipping to", DATA_DIR, "...")
    subprocess.run(["unzip", "-q", "-o", str(ZIP_PATH), "-d", str(DATA_DIR)], check=True)
print("data:", sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
%pip install -q tifffile scipy matplotlib pandas tqdm segmentation-models-pytorch timm
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("cwd:", os.getcwd())

In [ ]:
!python -m umod_unet.train --data-dir {DATA_DIR} --run-dir {RUN_DIR} --batch-size 16 --epochs 40

In [ ]:
import shutil
for name in ["results_epoch22.md", "val_confs.npy", "test_confs.npy", "examples_test.png"]:
    src = RUN_DIR / name
    shutil.copy(src, RUN_DIR / (src.stem + "_tta" + src.suffix))
!python -m umod_unet.train --data-dir {DATA_DIR} --run-dir {RUN_DIR} --batch-size 16 --epochs 40 --eval-only --no-tta

In [ ]:
import numpy as np

OLD = "/content/drive/MyDrive/umod_runs/torch_unet_v1/test_confs.npy"
names = ["background", "rod", "RBC/WBC", "yeast", "misc", "single EPC", "small EPC sheet", "large EPC sheet"]

def dice_per_class(conf):
    tp = np.diag(conf).astype(np.float64)
    gt, pr = conf.sum(1).astype(np.float64), conf.sum(0).astype(np.float64)
    return np.where(gt > 0, 2 * tp / np.maximum(gt + pr, 1), np.nan)

def compare(new_path, label, B=2000):
    old, new = np.load(OLD), np.load(new_path)
    assert old.shape == new.shape, (old.shape, new.shape)
    n = len(old)
    d_old, d_new = dice_per_class(old.sum(0)), dice_per_class(new.sum(0))
    idx = np.random.default_rng(0).integers(0, n, size=(B, n))   # the SAME resampled images for both models
    diff = np.empty((B, 8))
    for b in range(B):
        diff[b] = dice_per_class(new[idx[b]].sum(0)) - dice_per_class(old[idx[b]].sum(0))
    print("=== " + label + " minus old model | paired bootstrap over %d test images, %d resamples ===" % (n, B))
    for c in range(1, 8):
        lo, hi = np.nanpercentile(diff[:, c], [2.5, 97.5])
        print("%-16s old %.3f | new %.3f | difference %+.3f  [95%% CI %+.3f, %+.3f]" % (names[c], d_old[c], d_new[c], d_new[c] - d_old[c], lo, hi))
    macro = np.nanmean(diff[:, 1:], axis=1)
    lo, hi = np.percentile(macro, [2.5, 97.5])
    print("%-16s old %.3f | new %.3f | difference %+.3f  [95%% CI %+.3f, %+.3f]" % (
        "macro (1-7)", np.nanmean(d_old[1:]), np.nanmean(d_new[1:]), np.nanmean(d_new[1:]) - np.nanmean(d_old[1:]), lo, hi))
    print()

compare(RUN_DIR / "test_confs.npy", "new, NO TTA")
compare(RUN_DIR / "test_confs_tta.npy", "new, with TTA")